<a href="https://colab.research.google.com/github/Tabbymargaret/kamilimu_assignments/blob/main/sql/Kamilimu_assignments.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1. Setting up my environment

In [ ]:
import sqlite3
import pandas as pd


2. Loading datasets & establishing the connection

In [ ]:
# Load datasets
athletes = pd.read_csv("https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2021/2021-07-27/athlete_events.csv")
regions = pd.read_csv("https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2021/2021-07-27/noc_regions.csv")
sales = pd.read_csv("https://raw.githubusercontent.com/mwaskom/seaborn-data/master/tips.csv")
sales = pd.read_csv("https://raw.githubusercontent.com/justmarkham/DAT8/master/data/chipotle.tsv", sep='\t')

#establish connection
conn = sqlite3.connect(":memory:")

#save datasets as sql
athletes.to_sql("athletes_table", conn, index=False, if_exists="replace")
regions.to_sql("regions_table", conn,index=False, if_exists="replace")
sales.to_sql("sales_table", conn, index=False, if_exists="replace")

4622

3.**SECTION 1**

Use the athletes, regions, and sales datasets.

*3.1 The athletes dataset:*

In [ ]:
#Count the total number of medals won by each country and show the top 5.

pd.read_sql(
    """
    SELECT NOC, COUNT(Medal) As number_of_medals
    FROM athletes_table
    WHERE Medal != 'None'
    GROUP BY NOC
    ORDER BY number_of_medals DESC
    LIMIT 5
    """,conn,
)

,NOC,number_of_medals
0,USA,5637
1,URS,2503
2,GER,2165
3,GBR,2068
4,FRA,1777


In [ ]:
#Calculate the average age of athletes who won a Gold medal.
pd.read_sql(
    """
    SELECT AVG(Age) as Average_age
    FROM athletes_table
    WHERE Medal = 'Gold'
    """,conn
)

,Average_age
0,25.901013


In [ ]:
#How many distinct events are there in each sport?
pd.read_sql(
    """
    SELECT Sport, COUNT(DISTINCT Event) AS distinct_events
    FROM athletes_table
    GROUP BY Sport
    """,conn
)

,Sport,distinct_events
0,Aeronautics,1
1,Alpine Skiing,10
2,Alpinism,1
3,Archery,29
4,Art Competitions,29
...,...,...
61,Tug-Of-War,1
62,Volleyball,2
63,Water Polo,2
64,Weightlifting,21


In [ ]:
#Show all athletes from the United States (NOC = 'USA').
pd.read_sql(
    """
    SELECT DISTINCT Name, NOC
    FROM athletes_table
    WHERE NOC = 'USA'
    ORDER BY Name
    """,conn
)

,Name,NOC
0,A. M. Woods,USA
1,Aage Emil Brix,USA
2,Aarik Wilson,USA
3,Aaron Benjamin Herman,USA
4,Aaron Blunck,USA
...,...,...
9647,"Zantzinger, Borie & Medary",USA
9648,Zenon Snylyk,USA
9649,Zina Lynna Garrison (-Jackson),USA
9650,Zoe Ann Olsen-Jensen (-Bramham),USA


In [ ]:
#Count how many medals were awarded each year.
pd.read_sql(
    """
    SELECT Year, COUNT(Medal) As number_of_medals
    FROM athletes_table
    WHERE Medal != 'None'
    GROUP BY Year
    ORDER BY Year
    """,conn
)
#

,Year,number_of_medals
0,1896,143
1,1900,604
2,1904,486
3,1906,458
4,1908,831
5,1912,941
6,1920,1308
7,1924,962
8,1928,823
9,1932,739


In [ ]:
#Find all athlete records where height or weight is missing.

pd.read_sql(
    """
    SELECT Name, Height, Weight
    FROM athletes_table
    WHERE Height IS NULL OR Weight IS NULL
    """, conn
)

,Name,Height,Weight
0,Gunnar Nielsen Aaby,NaN,NaN
1,Edgar Lindenau Aabye,NaN,NaN
2,"Cornelia ""Cor"" Aalten (-Strannood)",168.0,NaN
3,"Cornelia ""Cor"" Aalten (-Strannood)",168.0,NaN
4,"Einar Ferdinand ""Einari"" Aalto",NaN,NaN
...,...,...,...
64258,Marius Edmund Zwiller,NaN,NaN
64259,Werner Zwingli,NaN,NaN
64260,Werner Zwingli,NaN,NaN
64261,Jan (Johann-) Zybert (Siebert-),NaN,NaN


In [ ]:
#Replace the missing height with the average athlete height.
pd.read_sql(
    """
    SELECT Name, Height,
           COALESCE(
               Height,
               (SELECT AVG(Height) FROM athletes_table)
           ) AS Cleaned_Height
    FROM athletes_table
    """, conn
)

,Name,Height,Cleaned_Height
0,A Dijiang,180.0,180.00000
1,A Lamusi,170.0,170.00000
2,Gunnar Nielsen Aaby,NaN,175.33897
3,Edgar Lindenau Aabye,NaN,175.33897
4,Christine Jacoba Aaftink,185.0,185.00000
...,...,...,...
271111,Andrzej ya,179.0,179.00000
271112,Piotr ya,176.0,176.00000
271113,Piotr ya,176.0,176.00000
271114,Tomasz Ireneusz ya,185.0,185.00000


*3.2 The sales dataset: where necessary, use the key word CAST to  remove the $ sign from the item_price*

In [ ]:
pd.read_sql(
    """
    SELECT COUNT(DISTINCT item_name)
    FROM sales_table
    """, conn
)

,COUNT(DISTINCT item_name)
0,50


In [ ]:
#Return total sales value per item.

pd.read_sql(
    """
    SELECT item_name, SUM(CAST(REPLACE(item_price, '$', '') AS REAL)) AS total_sales
    FROM sales_table
    GROUP BY item_name
    ORDER BY total_sales DESC
    LIMIT 5
    """, conn
)

,item_name,total_sales
0,Chicken Bowl,7342.73
1,Chicken Burrito,5575.82
2,Steak Burrito,3851.43
3,Steak Bowl,2260.19
4,Chips and Guacamole,2201.04


In [ ]:
pd.read_sql(
    """
    SELECT DISTINCT order_id AS unique_orders
    FROM sales_table
    """,conn
)

,unique_orders
0,1
1,2
2,3
3,4
4,5
...,...
1829,1830
1830,1831
1831,1832
1832,1833


**SECTION 2**

In [ ]:
df_high_performers = pd.read_sql(
    """
    WITH MedalistData AS (
        -- Nested query / CTE to find unique athletes who won at least one medal
        SELECT
            NOC,
            ID,
            -- SQLite handles AVG safely if we cast text Age or skip NULLs
            CAST(Age AS REAL) AS CleanedAge
        FROM athletes_table
        WHERE Medal IS NOT NULL
          AND Medal != 'NA'
          AND Medal != 'NaN'
          AND Age IS NOT NULL
          AND Age != 'NA'
          AND Age != 'NaN'
        GROUP BY NOC, ID
    )
    SELECT
        r.region AS Country,
        COUNT(m.ID) AS Total_Medalists,
        AVG(m.CleanedAge) AS Average_Age,
        CASE
            WHEN AVG(m.CleanedAge) < 25 THEN 'High'
            WHEN AVG(m.CleanedAge) BETWEEN 25 AND 30 THEN 'Medium'
            ELSE 'Low'
        -- Note: SQLite evaluates BETWEEN inclusively (25.0 to 30.0)
        END AS Performance
    FROM MedalistData m
    JOIN regions_table r ON m.NOC = r.NOC
    GROUP BY r.region
    HAVING Total_Medalists > 0
    ORDER BY Total_Medalists DESC;
    """, conn
)

print(df_high_performers.head(10))


       Country  Total_Medalists  Average_Age Performance
0          USA             3795    24.397101        High
1       Russia             2735    24.578062        High
2      Germany             2646    24.709373        High
3           UK             1503    27.267465      Medium
4       France             1201    26.819317      Medium
5       Sweden             1108    26.863718      Medium
6        Italy             1078    25.424861      Medium
7       Canada             1039    25.427334      Medium
8    Australia              883    24.267271        High
9  Netherlands              731    25.600547      Medium
